In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
district_flow_name = {
    1: 'Bellevue (excluding downtown)',
    2: 'Bellevue Downtown',
    3: 'Kirkland',
    4: 'Redmond',
    5: 'Seattle (excluding Seattle downtown)',
    6: 'Seattle downtown',
    7: 'Rest',
}
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Trip Arrival Times by Hour

In [3]:
def arr_time_by_hr(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 

    trip_ok_1 = data1['Trip'][['arrtm', 'trexpfac', 'travdist']].query('travdist > 0 and travdist < 200')
    trip_ok_3 = data3['Trip'][['arrtm', 'trexpfac', 'travdist']].query('travdist > 0 and travdist < 200')
    trip_ok_1 = trip_ok_1.reset_index()
    trip_ok_3 = trip_ok_3.reset_index()
    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()

    #Trip arrival time by hour
    trip_ok_1['hr'] = min_to_hour(trip_ok_1['arrtm'], 0)
    trip_ok_3['hr'] = min_to_hour(trip_ok_3['arrtm'], 0)
    trip_1_time = trip_ok_1[['hr', 'trexpfac']].groupby('hr').sum()['trexpfac']
    trip_3_time = trip_ok_3[['hr', 'trexpfac']].groupby('hr').sum()['trexpfac']
    trip_1_time_share = 100 * trip_1_time / trip_1_time.sum()
    trip_3_time_share = 100 * trip_3_time / trip_3_time.sum()
    trip_time = pd.DataFrame()
    trip_time[name1 + ' (%)'] = trip_1_time_share
    trip_time[name3 + ' (%)'] = trip_3_time_share
    trip_time = get_differences(trip_time, name1 + ' (%)', name3 + ' (%)', 2)
    trip_time = recode_index(trip_time, 'hr', 'Arrival Hour')

    trip_time = trip_time[[f'{name1} (%)', f'{name3} (%)', f'Difference ({name1} (%) - {name3} (%))']]

    display(trip_time.style.format({
        f'{name1} (%)': '{:,.1f}%',
        f'{name3} (%)': '{:,.1f}%',
        f'Difference ({name1} (%) - {name3} (%))': '{:,.1f}%',
    }))

    fig = px.bar(
        trip_time.reset_index(),
        x='Arrival Hour',
        y=[f'{name1} (%)', f'{name3} (%)'],
        barmode='group',
        labels={'index': 'Arrival Hour', 'value': 'Trip Arrival Time (hr)', 'variable': 'Source'},
        title=f'Trip Arrival Time (hr) ({tag})'
    )
    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Trip Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [4]:
arr_time_by_hr(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Arrival Hour,,,
00 - 01,0.4%,0.1%,0.3%
01 - 02,0.1%,0.1%,-0.0%
02 - 03,0.1%,0.1%,-0.0%
03 - 04,0.4%,0.1%,0.3%
04 - 05,0.6%,0.4%,0.2%
05 - 06,0.7%,0.9%,-0.3%
06 - 07,3.1%,2.3%,0.8%
07 - 08,5.8%,6.4%,-0.6%
08 - 09,7.3%,7.5%,-0.2%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
arr_time_by_hr(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='PSRC Region')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Arrival Hour,,,
00 - 01,0.3%,0.0%,0.3%
01 - 02,0.1%,0.0%,0.1%
02 - 03,0.1%,0.0%,0.1%
03 - 04,0.3%,0.1%,0.2%
04 - 05,0.5%,0.0%,0.5%
05 - 06,0.6%,0.9%,-0.3%
06 - 07,3.0%,2.1%,0.9%
07 - 08,5.8%,3.9%,2.0%
08 - 09,7.5%,9.8%,-2.3%


## Tour Primary Destination Arrival Times by Hour

In [6]:
def tour_pd_arr_time(data1=data_daysim, data2=data_survey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 

    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()
    tour_ok_2 = data2['Tour_cloned'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_2 = tour_ok_2.reset_index()

    #Tour Primary Destination arrival time by hour
    tour_ok_1['hrapd'] = min_to_hour(tour_ok_1['tardest'], 0)
    tour_ok_2['hrapd'] = min_to_hour(tour_ok_2['tardest'], 0)
    tour_1_time_apd = tour_ok_1[['hrapd', 'toexpfac']].groupby('hrapd').sum()['toexpfac']
    tour_2_time_apd = tour_ok_2[['hrapd', 'toexpfac']].groupby('hrapd').sum()['toexpfac']
    tour_1_time_share_apd = 100 * tour_1_time_apd / tour_1_time_apd.sum()
    tour_2_time_share_apd = 100 * tour_2_time_apd / tour_2_time_apd.sum()
    tour_time_apd = pd.DataFrame()
    tour_time_apd[name1 + ' (%)'] = tour_1_time_share_apd
    tour_time_apd[name2 + ' (%)'] = tour_2_time_share_apd
    tour_time_apd = get_differences(tour_time_apd, name1 + ' (%)', name2 + ' (%)', 2)
    tour_time_apd = recode_index(tour_time_apd, 'hrapd', 'Primary Destination Arrival Hour')
    tour_time_apd = tour_time_apd.loc[:, [f'{name1} (%)', f'{name2} (%)', f'Difference ({name1} (%) - {name2} (%))']]

    display(tour_time_apd.style.format({
        f'{name1} (%)': '{:,.1f}%',
        f'{name2} (%)': '{:,.1f}%',
        f'Difference ({name1} (%) - {name2} (%))': '{:,.1f}%',
    }))

    fig = px.bar(
        tour_time_apd.reset_index(),
        x='Primary Destination Arrival Hour',
        y=[f'{name1} (%)', f'{name2} (%)'],
        barmode='group',
        labels={'index': 'Primary Destination Arrival Hour', 'value': 'Tour Arrival Time (hr)', 'variable': 'Source'},
        title=f'Tour Arrival Time (hr) ({tag})'
    )

    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Tour Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [7]:
tour_pd_arr_time(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Primary Destination Arrival Hour,,,
00 - 01,0.1%,0.0%,0.0%
01 - 02,0.0%,0.0%,0.0%
02 - 03,0.0%,0.0%,0.0%
03 - 04,0.4%,0.0%,0.3%
04 - 05,0.5%,0.8%,-0.3%
05 - 06,0.6%,1.9%,-1.3%
06 - 07,5.1%,4.8%,0.3%
07 - 08,10.8%,11.5%,-0.7%
08 - 09,13.3%,14.2%,-1.0%


In [8]:
tour_pd_arr_time(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Primary Destination Arrival Hour,,,
00 - 01,0.0%,nan%,nan%
01 - 02,0.0%,nan%,nan%
02 - 03,0.0%,nan%,nan%
03 - 04,0.4%,0.0%,0.3%
04 - 05,0.5%,0.0%,0.4%
05 - 06,0.5%,0.2%,0.3%
06 - 07,4.9%,5.2%,-0.3%
07 - 08,10.7%,9.4%,1.2%
08 - 09,13.4%,20.5%,-7.2%


## Tour Primary Destination Departure Times by Hour

In [9]:
def tour_pd_depart_time(data1=data_daysim, data2=data_survey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 

    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()
    tour_ok_2 = data2['Tour_cloned'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_2 = tour_ok_2.reset_index()

    #Tour Primary Destination Departure time by hour
    tour_ok_1['hrlpd'] = min_to_hour(tour_ok_1['tlvdest'], 0)
    tour_ok_2['hrlpd'] = min_to_hour(tour_ok_2['tlvdest'], 0)
    tour_1_time_lpd = tour_ok_1[['hrlpd', 'toexpfac']].groupby('hrlpd').sum()['toexpfac']
    tour_2_time_lpd = tour_ok_2[['hrlpd', 'toexpfac']].groupby('hrlpd').sum()['toexpfac']
    tour_1_time_share_lpd = 100 * tour_1_time_lpd / tour_1_time_lpd.sum()
    tour_2_time_share_lpd = 100 * tour_2_time_lpd / tour_2_time_lpd.sum()
    tour_time_lpd = pd.DataFrame()
    tour_time_lpd[name1 + ' (%)'] = tour_1_time_share_lpd
    tour_time_lpd[name2 + ' (%)'] = tour_2_time_share_lpd
    tour_time_lpd = get_differences(tour_time_lpd, name1 + ' (%)', name2 + ' (%)', 2)
    tour_time_lpd = recode_index(tour_time_lpd, 'hrlpd', 'Primary Destination Departure Hour')
    tour_time_lpd = tour_time_lpd.loc[:, [f'{name1} (%)', f'{name2} (%)', f'Difference ({name1} (%) - {name2} (%))']]

    display(tour_time_lpd.style.format({
        f'{name1} (%)': '{:,.1f}%',
        f'{name2} (%)': '{:,.1f}%',
        f'Difference ({name1} (%) - {name2} (%))': '{:,.1f}%',
    }))

    fig = px.bar(
        tour_time_lpd.reset_index(),
        x='Primary Destination Departure Hour',
        y=[f'{name1} (%)', f'{name2} (%)'],
        barmode='group',
        labels={'index': 'Primary Destination Departure Hour', 'value': 'Tour Departure Time (hr)', 'variable': 'Source'},
        title=f'Tour Departure Time (hr) ({tag})'
    )

    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Tour Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [10]:
tour_pd_depart_time(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Primary Destination Departure Hour,,,
00 - 01,0.1%,0.1%,0.1%
01 - 02,0.1%,0.0%,0.1%
02 - 03,0.1%,0.0%,0.1%
03 - 04,0.1%,0.0%,0.1%
04 - 05,0.2%,0.1%,0.1%
05 - 06,0.2%,0.3%,-0.1%
06 - 07,0.4%,0.2%,0.2%
07 - 08,1.8%,1.9%,-0.1%
08 - 09,3.4%,3.7%,-0.3%


In [11]:
tour_pd_depart_time(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs (%),2023Survey (%),Difference (DaysimOutputs (%) - 2023Survey (%))
Primary Destination Departure Hour,,,
00 - 01,0.1%,nan%,nan%
01 - 02,0.1%,nan%,nan%
02 - 03,0.1%,nan%,nan%
03 - 04,0.1%,0.0%,0.1%
04 - 05,0.2%,nan%,nan%
05 - 06,0.2%,0.0%,0.1%
06 - 07,0.4%,nan%,nan%
07 - 08,1.8%,4.2%,-2.4%
08 - 09,3.4%,3.3%,0.1%


## Model Bike Trips Departure Time by Purpose

In [12]:
def bike_trip_depart_time_by_purp(data1=data_daysim, tag='PSRC Region'):    
    ### Create bike trip distribution by departure time and purpose
    bike_trips_df = data1['Trip'][['deptm', 'trexpfac', 'mode', 'dpurp']].query('mode == "Bike"').reset_index()
    bike_trips_df['hr'] = min_to_hour(bike_trips_df['deptm'], 0)
    bike_trips_by_hr_df = bike_trips_df[['hr', 'trexpfac']].groupby('hr').sum().reset_index()
    bike_trips_by_hr_df.rename(columns = {'hr':'Departure Hour', 'trexpfac':'All Purposes'}, inplace = True)
    trip_purposes = bike_trips_df['dpurp'].unique()
    for purp in trip_purposes:
        bike_trip_purp_by_hf_df = bike_trips_df.loc[bike_trips_df['dpurp'] == purp, ['hr', 'trexpfac']].groupby('hr').sum().reset_index()
        bike_trip_purp_by_hf_df.rename(columns = {'trexpfac':purp}, inplace = True)
        bike_trips_by_hr_df = bike_trips_by_hr_df.merge(bike_trip_purp_by_hf_df, left_on = 'Departure Hour', right_on = 'hr', how = 'outer')
        bike_trips_by_hr_df = bike_trips_by_hr_df.drop(columns = ['hr'])
    
    # table
    display(bike_trips_by_hr_df.style.format({col: '{:,.0f}' for col in bike_trips_by_hr_df.columns if col != 'Departure Hour'}))

In [13]:
bike_trip_depart_time_by_purp(data1=data_daysim, tag='PSRC Region')

,Departure Hour,All Purposes,Home,Social,Shop,Escort,Work,School,Meal,Personal Business
0,00 - 01,4,4,nan,nan,nan,nan,nan,nan,nan
1,01 - 02,4,4,nan,nan,nan,nan,nan,nan,nan
2,02 - 03,2,2,nan,nan,nan,nan,nan,nan,nan
3,03 - 04,4,nan,1,nan,3,nan,nan,nan,nan
4,04 - 05,74,nan,12,6,41,12,1,nan,2
5,05 - 06,270,nan,43,19,126,57,12,3,10
6,06 - 07,540,nan,121,39,212,108,30,12,18
7,07 - 08,"1,491",735,187,92,218,114,100,11,34
8,08 - 09,"2,132","1,352",193,129,189,71,129,23,46
9,09 - 10,"2,546","1,871",212,142,149,24,60,26,62


In [14]:
bike_trip_depart_time_by_purp(data1=data_daysim_bkr, tag='BKR')

,Departure Hour,All Purposes,Social,Home,School,Shop,Work,Escort,Meal,Personal Business
0,04 - 05,4,nan,nan,nan,nan,2,2,nan,nan
1,05 - 06,25,4,nan,nan,2,5,13,1,nan
2,06 - 07,80,20,nan,2,5,24,24,3,2
3,07 - 08,149,39,53,8,2,21,21,1,4
4,08 - 09,182,30,98,9,5,13,22,2,3
5,09 - 10,181,18,132,2,8,1,14,2,4
6,10 - 11,177,16,136,1,5,3,12,2,2
7,11 - 12,141,16,102,1,4,4,8,5,1
8,12 - 13,145,20,102,1,4,3,8,3,4
9,13 - 14,230,33,166,1,4,6,13,6,1


## Model Bike Tours Departure Time by Purpose

In [15]:
def bike_tour_depart_time_by_purp(data1=data_daysim, tag='PSRC Region'):
    ### create bike tour distribution by departure time and purpose
    bike_tours_df = data1['Tour'][['tlvorig', 'toexpfac', 'tmodetp', 'pdpurp']].query('tmodetp == "Bike"').reset_index()
    bike_tours_df['hrlvo'] = min_to_hour(bike_tours_df['tlvorig'], 0)
    bike_tours_by_hr_df = bike_tours_df[['hrlvo', 'toexpfac']].groupby('hrlvo').sum().reset_index()
    bike_tours_by_hr_df.rename(columns = {'hrlvo':'Departure Hour', 'toexpfac':'All Purposes'}, inplace = True)
    tour_purposes = bike_tours_df['pdpurp'].unique()
    for purp in tour_purposes:
        bike_tour_purp_by_hr_df = bike_tours_df.loc[bike_tours_df['pdpurp'] == purp, ['hrlvo', 'toexpfac']].groupby('hrlvo').sum().reset_index()
        bike_tour_purp_by_hr_df.rename(columns = {'toexpfac': purp}, inplace = True)
        bike_tours_by_hr_df = bike_tours_by_hr_df.merge(bike_tour_purp_by_hr_df, left_on = 'Departure Hour', right_on = 'hrlvo', how = 'outer')
        bike_tours_by_hr_df = bike_tours_by_hr_df.drop(columns = ['hrlvo'])
    
    # table
    display(bike_tours_by_hr_df.style.format({col: '{:,.0f}' for col in bike_tours_by_hr_df.columns if col != 'Departure Hour'}))

In [16]:
bike_tour_depart_time_by_purp(data1=data_daysim, tag='PSRC Region')

,Departure Hour,All Purposes,Social,Escort,School,Work,Shop,Meal,Personal Business
0,03 - 04,162,33,30,32,48,6,2,11
1,04 - 05,260,51,75,34,67,18,5,10
2,05 - 06,783,202,166,165,139,60,11,40
3,06 - 07,"2,439",633,264,983,245,174,29,111
4,07 - 08,"4,197","1,033",259,"2,053",250,393,42,167
5,08 - 09,"4,478","1,304",236,"2,015",148,499,49,227
6,09 - 10,"3,162","1,188",180,"1,015",64,434,57,224
7,10 - 11,"2,073",822,136,537,51,335,49,143
8,11 - 12,"1,915",850,164,342,43,333,45,138
9,12 - 13,"1,847",804,196,192,58,381,62,154


In [17]:
bike_tour_depart_time_by_purp(data1=data_daysim_bkr, tag='BKR')

,Departure Hour,All Purposes,Social,School,Work,Meal,Escort,Shop,Personal Business
0,03 - 04,22,3,2,13,nan,3,1,nan
1,04 - 05,30,4,2,18,1,5,nan,nan
2,05 - 06,98,37,15,15,3,19,5,4
3,06 - 07,203,57,65,41,5,24,4,7
4,07 - 08,290,121,87,35,1,22,16,8
5,08 - 09,248,127,52,20,4,23,13,9
6,09 - 10,164,92,25,4,5,17,11,10
7,10 - 11,133,72,15,11,6,8,13,8
8,11 - 12,128,79,7,5,4,11,16,6
9,12 - 13,134,82,2,7,10,10,17,6
